# Baseline Logistic Regression

This notebook builds the first fraud-detection baseline using the shared data, preprocessing, and evaluation modules. Model and threshold decisions are made on validation data only; the held-out test set remains untouched.

In [1]:
from pathlib import Path
import json
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import load_dataset, split_dataset
from src.evaluation import best_f1_threshold, classification_metrics
from src.models import build_logistic_regression
from src.preprocessing import build_linear_preprocessor, fit_transform_splits

In [2]:
dataset_path = project_root / 'data' / 'creditcard.csv'
transaction_dataset = load_dataset(dataset_path)

(
    training_features,
    validation_features,
    test_features,
    training_labels,
    validation_labels,
    test_labels,
) = split_dataset(transaction_dataset)

print(f'Train: {len(training_features):,} transactions')
print(f'Validation: {len(validation_features):,} transactions')
print(f'Test reserved: {len(test_features):,} transactions')
print(f'Training fraud rate: {training_labels.mean():.4%}')

Train: 170,884 transactions
Validation: 56,961 transactions
Test reserved: 56,962 transactions
Training fraud rate: 0.1726%


In [3]:
linear_preprocessor = build_linear_preprocessor(training_features.columns.tolist())
(
    transformed_training_features,
    transformed_validation_features,
    transformed_test_features,
    fitted_preprocessor,
) = fit_transform_splits(
    linear_preprocessor,
    training_features,
    validation_features,
    test_features,
)

In [4]:
logistic_model = build_logistic_regression()
logistic_model.fit(transformed_training_features, training_labels)

validation_fraud_scores = logistic_model.predict_proba(
    transformed_validation_features
)[:, 1]

selected_threshold = best_f1_threshold(
    validation_labels,
    validation_fraud_scores,
)
validation_metrics = classification_metrics(
    validation_labels,
    validation_fraud_scores,
    threshold=selected_threshold,
)
validation_metrics

{'threshold': 0.9999999980334442,
 'roc_auc': 0.9806087815074327,
 'pr_auc': 0.6858393374763166,
 'precision': 0.8604651162790697,
 'recall': 0.7551020408163265,
 'f1': 0.8043478260869565,
 'tn': 56851,
 'fp': 12,
 'fn': 24,
 'tp': 74}

In [5]:
metrics_directory = project_root / 'outputs' / 'metrics'
metrics_directory.mkdir(parents=True, exist_ok=True)
baseline_metrics_path = metrics_directory / 'logistic_regression_validation_metrics.json'

baseline_results = {
    'model': 'Logistic Regression',
    'selection_split': 'validation',
    **validation_metrics,
}

baseline_metrics_path.write_text(
    json.dumps(baseline_results, indent=2),
    encoding='utf-8',
)
print(f'Saved: {baseline_metrics_path.relative_to(project_root)}')

Saved: outputs/metrics/logistic_regression_validation_metrics.json


## Interpretation

The logistic model provides the reference point for later models. PR-AUC is the main discrimination metric because fraud is rare. The decision threshold is selected on validation data only. Final test evaluation is intentionally deferred until model and threshold selection are complete.